# AI-Powered Test Oracle: Data Fetching from Test Pipeline

This notebook fetches API and GraphQL test response data from the Instana test pipeline for analysis and assertion generation.

**Data Source**: http://reports-nfs.qe-auto-results.fyre.ibm.com/instana/reports/

**Path Pattern**: `preview+X.X.X/ → online/ → e2e/ → YYYYMMDD-HHMMSS/ → api/ → stan-api-*/`

## 1. Setup and Imports

In [ ]:
# Install required packages (run once)
!pip install requests beautifulsoup4 pandas lxml

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import re
from datetime import datetime
from urllib.parse import urljoin, urlparse
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("✓ All imports successful")

## 2. Configuration

In [ ]:
# Base URL for test reports
BASE_URL = "http://reports-nfs.qe-auto-results.fyre.ibm.com/instana/reports/"

# Save files in current workspace directory
DATA_DIR = Path("/Users/jisnyvarghese/a-i-powered-test-oracle-intelligent-assertion-generator-21a31ccd")

# Output files
OUTPUT_JSON = DATA_DIR / "test_responses.json"
OUTPUT_CSV = DATA_DIR / "test_responses.csv"

print(f"✓ Configuration set")
print(f"  Base URL: {BASE_URL}")
print(f"  Data directory: {DATA_DIR}")

## 3. Helper Functions

In [ ]:
def fetch_url(url, timeout=30):
    """
    Fetch content from URL with error handling
    """
    try:
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()
        return response
    except requests.exceptions.RequestException as e:
        print(f"❌ Error fetching {url}: {e}")
        return None

def parse_directory_listing(html_content, base_url):
    """
    Parse directory listing HTML to extract file/folder links
    """
    soup = BeautifulSoup(html_content, 'html.parser')
    links = []
    
    # Find all links in the directory listing
    for link in soup.find_all('a'):
        href = link.get('href')
        if href and href not in ['../', '../']:
            full_url = urljoin(base_url, href)
            links.append({
                'name': href.rstrip('/'),
                'url': full_url,
                'is_directory': href.endswith('/')
            })
    
    return links

def should_explore_directory(dir_name, current_depth, path_parts):
    """
    Determine if a directory should be explored based on the path pattern
    """
    dir_lower = dir_name.lower()
    
    # Level 0: At root level, only explore 'preview' directories
    if current_depth == 0:
        return 'preview' in dir_lower and '+' in dir_lower
    
    # Level 1: Explore 'online' directories
    if current_depth == 1:
        return dir_lower == 'online'
    
    # Level 2: Explore 'e2e' directories
    if current_depth == 2:
        return dir_lower == 'e2e'
    
    # Level 3: Explore date directories (format: YYYYMMDD-HHMMSS)
    if current_depth == 3:
        return re.match(r'\d{8}-\d{6}', dir_name) is not None
    
    # Level 4: Explore 'api' directories
    if current_depth == 4:
        return dir_lower == 'api'
    
    # Level 5: Explore stan-api-fast and stan-api-graphql directories
    if current_depth == 5:
        return 'stan-api-fast' in dir_lower or 'stan-api-graphql' in dir_lower
    
    return False

def classify_api_type(path):
    """
    Classify if the path contains REST API or GraphQL tests
    """
    path_lower = path.lower()
    if 'stan-api-graphql' in path_lower or 'graphql' in path_lower:
        return 'GraphQL'
    elif 'stan-api-fast' in path_lower or 'api' in path_lower:
        return 'REST'
    return 'Unknown'

print("✓ Helper functions defined")

## 4. Explore Directory Structure

In [ ]:
def explore_directory(url, max_depth=7, current_depth=0, path_parts=None):
    """
    Recursively explore directory structure following specific path patterns
    """
    if path_parts is None:
        path_parts = []
    
    if current_depth >= max_depth:
        return []
    
    indent = '  ' * current_depth
    dir_name = url.split('/')[-2] if url.endswith('/') else url.split('/')[-1]
    print(f"{indent}📁 Exploring (depth {current_depth}): {dir_name}")
    
    response = fetch_url(url)
    if not response:
        return []
    
    links = parse_directory_listing(response.text, url)
    all_files = []
    
    for link in links:
        if link['is_directory']:
            # Check if we should explore this directory
            if should_explore_directory(link['name'], current_depth, path_parts):
                new_path_parts = path_parts + [link['name']]
                sub_files = explore_directory(link['url'], max_depth, current_depth + 1, new_path_parts)
                all_files.extend(sub_files)
        else:
            # Only collect XML files in api test directories
            if link['name'].endswith('.xml'):
                api_type = classify_api_type(link['url'])
                print(f"{indent}  📄 Found [{api_type}]: {link['name']}")
                link['api_type'] = api_type
                link['path_parts'] = path_parts
                all_files.append(link)
    
    return all_files

# Explore the directory structure
print("\n🔍 Starting directory exploration...\n")
print("Following path pattern:")
print("  preview+X.X.X/ → online/ → e2e/ → YYYYMMDD-HHMMSS/ → api/ → stan-api-*/\n")

discovered_files = explore_directory(BASE_URL, max_depth=7)

print(f"\n✓ Discovery complete: Found {len(discovered_files)} test response files")

## 5. Display Discovered Files

In [ ]:
if discovered_files:
    df_files = pd.DataFrame(discovered_files)
    print(f"\n📊 Discovered Files Summary:")
    print(f"Total files: {len(df_files)}")
    
    # Count by API type
    api_counts = df_files['api_type'].value_counts()
    print(f"\nAPI Type Distribution:")
    for api_type, count in api_counts.items():
        print(f"  {api_type}: {count} files")
    
    display(df_files[['name', 'api_type', 'url']].head(20))
else:
    print("⚠️ No files discovered. The directory might be empty or require authentication.")

## 6. Fetch Test Response Data

In [ ]:
def fetch_test_responses(file_list, max_files=100):
    """
    Fetch content from test response files
    """
    test_data = []
    
    total_files = min(len(file_list), max_files)
    print(f"\n⏳ Fetching {total_files} test response files...\n")
    
    for i, file_info in enumerate(file_list[:max_files]):
        print(f"  [{i+1}/{total_files}] Fetching: {file_info['name']}")
        
        response = fetch_url(file_info['url'])
        if not response:
            continue
        
        content_type = response.headers.get('Content-Type', '')
        
        data_entry = {
            'filename': file_info['name'],
            'url': file_info['url'],
            'api_type': file_info.get('api_type', 'Unknown'),
            'content_type': content_type,
            'size_bytes': len(response.content),
            'fetched_at': datetime.now().isoformat(),
            'path_parts': file_info.get('path_parts', [])
        }
        
        # Parse XML content
        try:
            data_entry['content'] = response.text
            data_entry['format'] = 'xml'
        except Exception as e:
            data_entry['content'] = response.text
            data_entry['format'] = 'text'
            data_entry['parse_error'] = str(e)
        
        test_data.append(data_entry)
    
    print(f"\n✓ Fetched {len(test_data)} test response files")
    return test_data

# Fetch test responses
if discovered_files:
    test_responses = fetch_test_responses(discovered_files, max_files=100)
else:
    test_responses = []
    print("⚠️ No files to fetch")

## 7. Analyze Test Response Data

In [ ]:
def analyze_test_responses(responses):
    """
    Analyze fetched test responses
    """
    if not responses:
        print("⚠️ No responses to analyze")
        return
    
    print("\n📊 Test Response Analysis:\n")
    
    # Format distribution
    formats = {}
    for resp in responses:
        fmt = resp.get('format', 'unknown')
        formats[fmt] = formats.get(fmt, 0) + 1
    
    print("Format Distribution:")
    for fmt, count in formats.items():
        print(f"  {fmt}: {count} files")
    
    # Size statistics
    sizes = [resp['size_bytes'] for resp in responses]
    print(f"\nSize Statistics:")
    print(f"  Total size: {sum(sizes):,} bytes ({sum(sizes)/1024/1024:.2f} MB)")
    print(f"  Average size: {sum(sizes)/len(sizes):,.0f} bytes")
    print(f"  Min size: {min(sizes):,} bytes")
    print(f"  Max size: {max(sizes):,} bytes")
    
    # API type distribution
    api_types = {}
    for resp in responses:
        api_type = resp.get('api_type', 'Unknown')
        api_types[api_type] = api_types.get(api_type, 0) + 1
    
    print(f"\nAPI Type Distribution:")
    for api_type, count in api_types.items():
        print(f"  {api_type}: {count} files")

# Analyze the data
analyze_test_responses(test_responses)

## 8. Save Data to Files

In [ ]:
def save_test_data(responses, json_path, csv_path):
    """
    Save test response data to JSON and CSV files
    """
    if not responses:
        print("⚠️ No data to save")
        return
    
    # Save to JSON
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(responses, f, indent=2, ensure_ascii=False)
    print(f"✓ Saved JSON data to: {json_path}")
    
    # Create CSV-friendly version (without nested content)
    csv_data = []
    for resp in responses:
        csv_row = {
            'filename': resp['filename'],
            'url': resp['url'],
            'api_type': resp.get('api_type', 'Unknown'),
            'format': resp.get('format', 'unknown'),
            'content_type': resp.get('content_type', ''),
            'size_bytes': resp['size_bytes'],
            'fetched_at': resp['fetched_at'],
            'has_content': bool(resp.get('content')),
            'parse_error': resp.get('parse_error', '')
        }
        csv_data.append(csv_row)
    
    df = pd.DataFrame(csv_data)
    df.to_csv(csv_path, index=False)
    print(f"✓ Saved CSV metadata to: {csv_path}")
    
    return df

# Save the data
if test_responses:
    df_summary = save_test_data(test_responses, OUTPUT_JSON, OUTPUT_CSV)
    print(f"\n📁 Data saved successfully!")
    print(f"  JSON: {OUTPUT_JSON}")
    print(f"  CSV: {OUTPUT_CSV}")
else:
    print("⚠️ No data to save")

## 9. Extract and Save API-Specific Data

In [ ]:
def extract_api_graphql_data(responses):
    """
    Separate and structure REST API and GraphQL responses
    """
    rest_api_data = []
    graphql_data = []
    
    for resp in responses:
        api_type = resp.get('api_type', 'Unknown')
        
        if api_type == 'GraphQL':
            graphql_data.append({
                'filename': resp['filename'],
                'url': resp['url'],
                'type': 'GraphQL',
                'content': resp.get('content', ''),
                'size': resp['size_bytes']
            })
        elif api_type == 'REST':
            rest_api_data.append({
                'filename': resp['filename'],
                'url': resp['url'],
                'type': 'REST',
                'content': resp.get('content', ''),
                'size': resp['size_bytes']
            })
    
    return rest_api_data, graphql_data

if test_responses:
    rest_data, graphql_data = extract_api_graphql_data(test_responses)
    
    print(f"\n🔍 API Type Classification:")
    print(f"  REST API responses: {len(rest_data)}")
    print(f"  GraphQL responses: {len(graphql_data)}")
    
    # Save separated data
    if rest_data:
        rest_file = DATA_DIR / "rest_api_responses.json"
        with open(rest_file, 'w') as f:
            json.dump(rest_data, f, indent=2)
        print(f"  ✓ REST API data saved to: {rest_file}")
    
    if graphql_data:
        graphql_file = DATA_DIR / "graphql_responses.json"
        with open(graphql_file, 'w') as f:
            json.dump(graphql_data, f, indent=2)
        print(f"  ✓ GraphQL data saved to: {graphql_file}")

## 10. Display Sample Data

In [ ]:
if test_responses:
    print("\n📋 Sample Test Response Data:\n")
    
    # Display first response
    sample = test_responses[0]
    print(f"Filename: {sample['filename']}")
    print(f"API Type: {sample.get('api_type', 'Unknown')}")
    print(f"Format: {sample.get('format', 'unknown')}")
    print(f"Size: {sample['size_bytes']:,} bytes")
    print(f"\nContent Preview (first 500 chars):")
    
    content = sample.get('content', '')
    print(str(content)[:500] + "...")
    
    # Display summary table
    if 'df_summary' in locals():
        print("\n\n📊 Summary Table:")
        display(df_summary.head(10))
else:
    print("⚠️ No data available to display")

## Summary

This notebook provides:

✅ **Smart Navigation**: Follows the exact directory structure pattern  
✅ **Preview Filtering**: Only explores directories with 'preview' in the name  
✅ **API Classification**: Automatically identifies REST vs GraphQL tests  
✅ **XML Collection**: Fetches all XML test response files  
✅ **Data Organization**: Separates REST API and GraphQL responses  
✅ **Local Storage**: Saves all files in the workspace directory  

**Output Files** (in workspace directory):
- `test_responses.json` - All responses with metadata
- `rest_api_responses.json` - REST API responses only
- `graphql_responses.json` - GraphQL responses only
- `test_responses.csv` - Summary metadata

**Next Steps:**
1. Review the fetched data
2. Use this data to train/prompt your AI model
3. Generate intelligent assertions based on patterns
4. Integrate with your test framework